In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# -------------------------------
# Configuration
# -------------------------------
TRAIN_PATH = '/kaggle/input/datasets/ashishmotwani/tomato/train'
EXTERNAL_PATH = '/kaggle/input/datasets/ashishmotwani/tomato/valid'

IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff'}

def analyze_dataset(root_path, dataset_name):
    """Walk through root_path, collect info per class."""
    root = Path(root_path)
    if not root.exists():
        print(f"Path {root_path} does not exist.")
        return None
    
    # Collect data
    data = []  # list of dicts per image
    class_counts = Counter()
    extension_counter = Counter()
    size_stats = {'width': [], 'height': []}
    mode_counter = Counter()  # RGB, L, etc.
    corrupt_files = []
    
    # Get class directories
    class_dirs = [d for d in root.iterdir() if d.is_dir()]
    print(f"\n{'='*60}")
    print(f"Analyzing {dataset_name} - found {len(class_dirs)} class directories")
    
    for class_dir in sorted(class_dirs):
        class_name = class_dir.name
        print(f"  Processing class: {class_name}")
        for img_path in class_dir.iterdir():
            if not img_path.is_file():
                continue
            ext = img_path.suffix.lower()
            if ext not in IMG_EXTENSIONS:
                extension_counter[ext] += 1  # count non-image files
                continue
            
            # Try to open with PIL
            try:
                with Image.open(img_path) as img:
                    width, height = img.size
                    mode = img.mode
                    # Verify by loading data
                    img.load()  # forces full load, may raise
                
                # If successful, record
                data.append({
                    'class': class_name,
                    'path': str(img_path),
                    'ext': ext,
                    'width': width,
                    'height': height,
                    'mode': mode
                })
                class_counts[class_name] += 1
                extension_counter[ext] += 1
                size_stats['width'].append(width)
                size_stats['height'].append(height)
                mode_counter[mode] += 1
            except Exception as e:
                corrupt_files.append(str(img_path))
                # Optionally print first few errors
                if len(corrupt_files) <= 5:
                    print(f"    Corrupt file: {img_path.name} - {e}")
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Summary
    print(f"\n{dataset_name} Summary:")
    print(f"  Total valid images: {len(df)}")
    print(f"  Corrupt/unreadable images: {len(corrupt_files)}")
    print(f"  Number of classes: {df['class'].nunique() if len(df) > 0 else 0}")
    print(f"  Class distribution:")
    for cls, cnt in class_counts.most_common():
        print(f"    {cls}: {cnt}")
    
    print(f"  File extensions:")
    for ext, cnt in extension_counter.most_common():
        print(f"    {ext if ext else 'no extension'}: {cnt}")
    
    print(f"  Image modes:")
    for mode, cnt in mode_counter.most_common():
        print(f"    {mode}: {cnt}")
    
    if size_stats['width']:
        print(f"  Width: min={min(size_stats['width'])}, max={max(size_stats['width'])}, mean={np.mean(size_stats['width']):.1f}")
        print(f"  Height: min={min(size_stats['height'])}, max={max(size_stats['height'])}, mean={np.mean(size_stats['height']):.1f}")
    
    return df, class_counts, size_stats, mode_counter, corrupt_files

# Run analysis
train_df, train_counts, train_sizes, train_modes, train_corrupt = analyze_dataset(TRAIN_PATH, "TRAIN SET")
ext_df, ext_counts, ext_sizes, ext_modes, ext_corrupt = analyze_dataset(EXTERNAL_PATH, "EXTERNAL SET")

# -------------------------------
# Visualizations
# -------------------------------
def plot_class_distribution(class_counts, title, save_name):
    plt.figure(figsize=(12,6))
    classes = list(class_counts.keys())
    counts = list(class_counts.values())
    plt.bar(classes, counts, color='skyblue')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Number of Images')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(save_name, dpi=150)
    plt.show()

def plot_size_distribution(sizes, title, save_name):
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    axes[0].hist(sizes['width'], bins=30, alpha=0.7, color='green')
    axes[0].set_xlabel('Width (pixels)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Width Distribution')
    axes[1].hist(sizes['height'], bins=30, alpha=0.7, color='orange')
    axes[1].set_xlabel('Height (pixels)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Height Distribution')
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_name, dpi=150)
    plt.show()

def plot_extension_pie(ext_counter, title, save_name):
    labels = [f"{ext} ({cnt})" for ext, cnt in ext_counter.most_common()]
    sizes = [cnt for ext, cnt in ext_counter.most_common()]
    plt.figure(figsize=(8,8))
    plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140)
    plt.title(title)
    plt.axis('equal')
    plt.tight_layout()
    plt.savefig(save_name, dpi=150)
    plt.show()

# Generate plots if data exists
if train_df is not None and len(train_df) > 0:
    plot_class_distribution(train_counts, "Train Set Class Distribution", "train_class_dist.png")
    plot_size_distribution(train_sizes, "Train Set Image Size Distribution", "train_size_dist.png")
    plot_extension_pie(Counter(train_df['ext']), "Train Set File Extensions", "train_ext_pie.png")

if ext_df is not None and len(ext_df) > 0:
    plot_class_distribution(ext_counts, "External Set Class Distribution", "ext_class_dist.png")
    plot_size_distribution(ext_sizes, "External Set Image Size Distribution", "ext_size_dist.png")
    plot_extension_pie(Counter(ext_df['ext']), "External Set File Extensions", "ext_ext_pie.png")

# Also print the mapping from external class names to train indices as previously found
print("\nClass mapping from external to train (based on naming):")
print("External Class -> Train Class (index)")
for ext_dir in Path(EXTERNAL_PATH).iterdir():
    if ext_dir.is_dir():
        ext_name = ext_dir.name
        # find matching train class (simplified heuristic)
        matched = False
        for train_name in train_counts.keys():
            # clean both
            clean_train = train_name.replace('Tomato___', '').replace('_', ' ').strip().lower()
            clean_ext = ext_name.replace('_', ' ').strip().lower()
            if clean_ext in clean_train or clean_train in clean_ext:
                print(f"  {ext_name} -> {train_name}")
                matched = True
                break
        if not matched:
            print(f"  {ext_name} -> (no match)")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import time
import warnings
import pathlib
import joblib
from PIL import Image
from tqdm import tqdm
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import layers

import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.calibration import calibration_curve
from sklearn.utils import resample
from sklearn.manifold import TSNE
import shap
from scipy.optimize import minimize

# -------------------------------
# 0. Create output directories
# -------------------------------
output_dir = pathlib.Path("outputs")
plots_dir = output_dir / "plots"
tables_dir = output_dir / "tables"
models_dir = output_dir / "models"
for d in [plots_dir, tables_dir, models_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print("XGBoost version:", xgb.__version__)

# -------------------------------
# 1. Paths and Parameters
# -------------------------------
TRAIN_PATH = '/kaggle/input/datasets/ashishmotwani/tomato/train'
EXTERNAL_PATH = '/kaggle/input/datasets/ashishmotwani/tomato/valid'

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42
VALIDATION_SPLIT = 0.2

# Only extensions that tf.image.decode_image can handle
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}

tf.random.set_seed(SEED)
np.random.seed(SEED)

# -------------------------------
# 2. Helper: verify image with TensorFlow decoder
# -------------------------------
def is_tf_decodable(file_path):
    try:
        img_str = tf.io.read_file(file_path)
        _ = tf.image.decode_image(img_str, channels=3, expand_animations=False)
        return True
    except:
        return False

# -------------------------------
# 3. Robust Loading of Training Data with PIL + TF verification
# -------------------------------
train_path = pathlib.Path(TRAIN_PATH)
class_dirs = [d for d in train_path.iterdir() if d.is_dir()]
class_names = sorted([d.name for d in class_dirs])
class_to_idx = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

valid_image_paths = []
valid_labels = []

for class_dir in class_dirs:
    class_name = class_dir.name
    class_idx = class_to_idx[class_name]
    print(f"Processing class: {class_name}")
    for img_path in tqdm(list(class_dir.iterdir()), desc=f"  {class_name}"):
        if img_path.suffix.lower() not in VALID_EXTENSIONS:
            continue
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception:
            continue
        if not is_tf_decodable(str(img_path)):
            print(f"  Skipping file not decodable by TF: {img_path.name}")
            continue
        valid_image_paths.append(str(img_path))
        valid_labels.append(class_idx)

print(f"\nTotal valid training images: {len(valid_image_paths)}")
print(f"Class distribution: {np.bincount(valid_labels)}")

# -------------------------------
# 4. Load External Dataset (same robust method)
# -------------------------------
external_path = pathlib.Path(EXTERNAL_PATH)
all_class_dirs = [d for d in external_path.iterdir() if d.is_dir()]
print("External class directories:", [d.name for d in all_class_dirs])

label_map = {}
unmapped_classes = []
for ext_dir in all_class_dirs:
    ext_name = ext_dir.name
    matched = False
    for idx, train_name in enumerate(class_names):
        clean_train = train_name.replace('Tomato___', '').replace('_', ' ').strip().lower()
        clean_ext = ext_name.replace('_', ' ').strip().lower()
        if clean_ext in clean_train or clean_train in clean_ext:
            label_map[ext_name] = idx
            print(f"Mapped '{ext_name}' -> '{train_name}' (index {idx})")
            matched = True
            break
    if not matched:
        unmapped_classes.append(ext_name)
        print(f"Warning: No mapping for external class '{ext_name}'. It will be ignored.")

valid_ext_paths = []
valid_ext_labels = []

for ext_dir in all_class_dirs:
    ext_name = ext_dir.name
    if ext_name not in label_map:
        continue
    class_label = label_map[ext_name]
    print(f"Processing external class: {ext_name}")
    for img_path in tqdm(list(ext_dir.iterdir()), desc=f"  {ext_name}"):
        if img_path.suffix.lower() not in VALID_EXTENSIONS:
            continue
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception:
            continue
        if not is_tf_decodable(str(img_path)):
            print(f"  Skipping file not decodable by TF: {img_path.name}")
            continue
        valid_ext_paths.append(str(img_path))
        valid_ext_labels.append(class_label)

print(f"\nTotal valid external images: {len(valid_ext_paths)}")
print(f"Class distribution in external set: {np.bincount(valid_ext_labels)}")

# -------------------------------
# 5. Build Feature Extractor (MobileNetV2)
# -------------------------------
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

feature_extractor = tf.keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D()
])

# Save the feature extractor model
try:
    feature_extractor.save(models_dir / "feature_extractor.keras")
    print("Feature extractor saved.")
except Exception as e:
    print(f"Could not save feature extractor: {e}")

# -------------------------------
# 6. Helper to create dataset and extract features
# -------------------------------
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)
    return image, label

def create_dataset(paths, labels, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def extract_features(dataset, model, verbose=True):
    features, labels = [], []
    total_batches = tf.data.experimental.cardinality(dataset).numpy()
    for i, (images, lbls) in enumerate(dataset):
        if verbose and i % 10 == 0:
            print(f"Processing batch {i+1}/{total_batches}")
        images_pp = preprocess_input(images)
        feats = model.predict(images_pp, verbose=0)
        features.append(feats)
        labels.append(lbls.numpy())
    return np.vstack(features), np.concatenate(labels)

# -------------------------------
# 7. Extract features for ALL training images (for cross‑validation)
# -------------------------------
full_train_ds = create_dataset(valid_image_paths, valid_labels, BATCH_SIZE, shuffle=False)
print("Extracting features for full training set...")
X_full, y_full = extract_features(full_train_ds, feature_extractor)
print(f"Full training features shape: {X_full.shape}")

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_full, y_full,
    test_size=VALIDATION_SPLIT,
    random_state=SEED,
    stratify=y_full
)
print(f"Training features: {X_train.shape}, Validation features: {X_val.shape}")

# Also keep the path splits for handcrafted feature extraction
X_train_paths, X_val_paths, y_train_paths, y_val_paths = train_test_split(
    valid_image_paths, valid_labels,
    test_size=VALIDATION_SPLIT,
    random_state=SEED,
    stratify=valid_labels
)

# -------------------------------
# 8. Extract features for external set
# -------------------------------
external_ds = create_dataset(valid_ext_paths, valid_ext_labels, BATCH_SIZE, shuffle=False)
print("Extracting external validation features...")
X_ext, y_ext = extract_features(external_ds, feature_extractor)
print(f"External features shape: {X_ext.shape}")

# -------------------------------
# 9. Variant C: MobileNetV2 + XGBoost (Original Hybrid)
# -------------------------------
print("\n--- Variant C: MobileNetV2 + XGBoost ---")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=num_classes,
    random_state=SEED,
    use_label_encoder=False,
    eval_metric='mlogloss',
    early_stopping_rounds=10
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

# Save XGBoost model
try:
    xgb_model.save_model(models_dir / "xgb_model.json")
    joblib.dump(xgb_model, models_dir / "xgb_model.pkl")
    print("XGBoost model saved as .json and .pkl")
except Exception as e:
    print(f"Could not save XGBoost model: {e}")

y_pred_ext = xgb_model.predict(X_ext)
y_proba_ext = xgb_model.predict_proba(X_ext)
acc_c = accuracy_score(y_ext, y_pred_ext)
print(f"Variant C Accuracy: {acc_c:.4f}")

# -------------------------------
# 10. Variant A: MobileNetV2 + Softmax
# -------------------------------
print("\n--- Variant A: MobileNetV2 + Softmax ---")
try:
    # Build a simple classifier head on the extracted features
    input_layer = tf.keras.Input(shape=(1280,))
    x = tf.keras.layers.Dropout(0.5)(input_layer)
    output_layer = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    cnn_head_model = tf.keras.Model(inputs=input_layer, outputs=output_layer)

    cnn_head_model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Train on the same extracted features
    history = cnn_head_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        verbose=0,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
    )

    # Evaluate on external set
    cnn_proba = cnn_head_model.predict(X_ext, verbose=0)
    cnn_pred = np.argmax(cnn_proba, axis=1)
    acc_a = accuracy_score(y_ext, cnn_pred)
    ece_a = compute_ece(y_ext, cnn_proba)   # compute_ece defined later, but we'll call after its definition
    print(f"Variant A Accuracy: {acc_a:.4f}")

    # Save the model
    cnn_head_model.save(models_dir / "softmax_head.keras")
    print("Softmax head model saved.")

except Exception as e:
    print(f"Variant A failed: {e}")
    acc_a = None

# -------------------------------
# 11. Variant B: Handcrafted Features + XGBoost
# -------------------------------
print("\n--- Variant B: Handcrafted Features + XGBoost ---")
try:
    def extract_handcrafted_features(img_path):
        """Extract color histograms (HSV) and global mean/std."""
        img = cv2.imread(img_path)
        if img is None:
            return None
        img = cv2.resize(img, (224, 224))
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

        # Color histograms (3 channels * 8 bins)
        hist_h = cv2.calcHist([hsv], [0], None, [8], [0, 180]).flatten()
        hist_s = cv2.calcHist([hsv], [1], None, [8], [0, 256]).flatten()
        hist_v = cv2.calcHist([hsv], [2], None, [8], [0, 256]).flatten()

        # Global stats (3 channels * 2 stats)
        mean = np.mean(img, axis=(0, 1))
        std = np.std(img, axis=(0, 1))

        return np.concatenate([hist_h, hist_s, hist_v, mean, std])   # 8+8+8+3+3 = 30 features

    # Extract handcrafted features for training, validation, and external
    def extract_handcrafted_batch(paths, desc="Extracting handcrafted"):
        feats = []
        valid_paths = []
        for p in tqdm(paths, desc=desc):
            f = extract_handcrafted_features(p)
            if f is not None:
                feats.append(f)
                valid_paths.append(p)
            else:
                print(f"Warning: could not extract handcrafted from {p}")
        return np.array(feats), valid_paths

    X_train_hand, train_paths_ok = extract_handcrafted_batch(X_train_paths, "Train handcrafted")
    X_val_hand, val_paths_ok = extract_handcrafted_batch(X_val_paths, "Val handcrafted")
    X_ext_hand, ext_paths_ok = extract_handcrafted_batch(valid_ext_paths, "External handcrafted")

    # Align labels with successfully extracted paths (they should be in same order)
    # We can use the original labels if we kept the paths; simpler: rebuild label arrays from the paths
    def get_labels_for_paths(paths, all_paths, all_labels):
        # Create mapping from path to label
        path_to_label = {p: l for p, l in zip(all_paths, all_labels)}
        return np.array([path_to_label[p] for p in paths])

    y_train_hand = get_labels_for_paths(train_paths_ok, X_train_paths, y_train)
    y_val_hand = get_labels_for_paths(val_paths_ok, X_val_paths, y_val)
    y_ext_hand = get_labels_for_paths(ext_paths_ok, valid_ext_paths, valid_ext_labels)

    # Train XGBoost on handcrafted features
    xgb_hand = xgb.XGBClassifier(
        n_estimators=200, max_depth=6,
        random_state=SEED, use_label_encoder=False,
        eval_metric='mlogloss'
    )
    xgb_hand.fit(
        X_train_hand, y_train_hand,
        eval_set=[(X_val_hand, y_val_hand)],
        verbose=False
    )

    # Save model
    joblib.dump(xgb_hand, models_dir / "xgb_handcrafted.pkl")
    xgb_hand.save_model(models_dir / "xgb_handcrafted.json")

    # Predict on external
    y_pred_hand = xgb_hand.predict(X_ext_hand)
    acc_b = accuracy_score(y_ext_hand, y_pred_hand)
    print(f"Variant B Accuracy: {acc_b:.4f}")

    # Compute ECE for handcrafted (needs probabilities)
    y_proba_hand = xgb_hand.predict_proba(X_ext_hand)
    ece_b = compute_ece(y_ext_hand, y_proba_hand)
    print(f"Variant B ECE: {ece_b:.4f}")

except Exception as e:
    print(f"Variant B failed: {e}")
    acc_b = None
    ece_b = None

# -------------------------------
# 12. Define ECE function (used in multiple places)
# -------------------------------
def compute_ece(y_true, y_proba, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    confidences = np.max(y_proba, axis=1)
    predictions = np.argmax(y_proba, axis=1)
    accuracies = (predictions == y_true).astype(float)
    ece = 0.0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        if prop_in_bin > 0:
            accuracy_in_bin = np.mean(accuracies[in_bin])
            confidence_in_bin = np.mean(confidences[in_bin])
            ece += np.abs(accuracy_in_bin - confidence_in_bin) * prop_in_bin
    return ece

# Compute ECE for Variant C (original hybrid)
ece_c = compute_ece(y_ext, y_proba_ext)
print(f"Variant C ECE: {ece_c:.4f}")

# -------------------------------
# 13. Variant D: Hybrid with Feature Pruning (Uncertainty-Aware)
# -------------------------------
print("\n--- Variant D: Hybrid Pruned ---")
try:
    # Compute feature stability using TTA on a subset of external images
    feature_stds = []
    n_samples_sel = 100
    sample_imgs = []
    for imgs, _ in external_ds.take(1):
        sample_imgs = imgs.numpy()[:n_samples_sel]
        break

    for img in sample_imgs:
        img_pp = preprocess_input(img)
        aug_images = []
        for _ in range(10):
            aug = tf.image.random_flip_left_right(img_pp)
            aug = tf.image.random_brightness(aug, max_delta=0.1)
            aug_images.append(aug)
        aug_images = tf.stack(aug_images)
        feats = feature_extractor.predict(aug_images, verbose=0)
        feat_std = np.std(feats, axis=0)
        feature_stds.append(feat_std)

    mean_feature_std = np.mean(feature_stds, axis=0)
    # Keep features with std below 90th percentile (most stable)
    stable_mask = mean_feature_std < np.percentile(mean_feature_std, 90)
    print(f"Stable features: {np.sum(stable_mask)}/{len(mean_feature_std)}")

    # Prune feature matrices
    X_train_pruned = X_train[:, stable_mask]
    X_val_pruned = X_val[:, stable_mask]
    X_ext_pruned = X_ext[:, stable_mask]

    # Train new XGBoost on pruned features
    xgb_pruned = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        objective='multi:softprob', num_class=num_classes,
        random_state=SEED, use_label_encoder=False,
        eval_metric='mlogloss', early_stopping_rounds=10
    )
    xgb_pruned.fit(
        X_train_pruned, y_train,
        eval_set=[(X_val_pruned, y_val)],
        verbose=False
    )

    # Save pruned model and mask
    np.save(output_dir / "stable_feature_mask.npy", stable_mask)
    xgb_pruned.save_model(models_dir / "xgb_pruned.json")
    joblib.dump(xgb_pruned, models_dir / "xgb_pruned.pkl")

    # Evaluate
    y_pred_pruned = xgb_pruned.predict(X_ext_pruned)
    y_proba_pruned = xgb_pruned.predict_proba(X_ext_pruned)
    acc_d = accuracy_score(y_ext, y_pred_pruned)
    ece_d = compute_ece(y_ext, y_proba_pruned)
    print(f"Variant D Accuracy: {acc_d:.4f}, ECE: {ece_d:.4f}")

except Exception as e:
    print(f"Variant D failed: {e}")
    stable_mask = None
    acc_d = None
    ece_d = None

# -------------------------------
# 14. Compile Ablation Results Table
# -------------------------------
ablation_data = {
    "Variant": ["A (CNN+Softmax)", "B (Handcrafted+XGB)", "C (Hybrid Ours)", "D (Hybrid Pruned)"],
    "Accuracy": [acc_a if acc_a is not None else None, acc_b if acc_b is not None else None, acc_c, acc_d if acc_d is not None else None],
    "ECE": [ece_a if 'ece_a' in locals() else None, ece_b if ece_b is not None else None, ece_c, ece_d if ece_d is not None else None],
    "Num Features": [1280, 30, 1280, np.sum(stable_mask) if stable_mask is not None else None]
}
ablation_df = pd.DataFrame(ablation_data)
ablation_df.to_csv(tables_dir / "ablation_study.csv", index=False)
print("\nAblation results saved to CSV.")

# -------------------------------
# 15. Plot Ablation Comparison Bar Chart
# -------------------------------
try:
    plt.figure(figsize=(10,6))
    x = np.arange(len(ablation_df))
    width = 0.35
    acc_vals = ablation_df["Accuracy"].fillna(0).values
    ece_vals = ablation_df["ECE"].fillna(0).values
    plt.bar(x - width/2, acc_vals, width, label='Accuracy', color='skyblue')
    plt.bar(x + width/2, ece_vals, width, label='ECE', color='salmon')
    plt.xlabel('Variant')
    plt.ylabel('Score')
    plt.title('Ablation Study: Accuracy vs. Calibration Error')
    plt.xticks(x, ablation_df["Variant"], rotation=15)
    plt.legend()
    plt.tight_layout()
    plt.savefig(plots_dir / "ablation_comparison.png", dpi=150)
    plt.show()
except Exception as e:
    print(f"Could not plot ablation chart: {e}")

# -------------------------------
# 16. Evaluation on External Set (original detailed metrics)
# -------------------------------
print("\n--- Detailed Evaluation of Variant C (Hybrid) ---")
print(f"External Validation Accuracy: {acc_c:.4f}")

# Classification report as DataFrame and CSV
report_dict = classification_report(y_ext, y_pred_ext, target_names=[class_names[i] for i in np.unique(y_ext)], output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(tables_dir / "classification_report.csv")
print("Classification report saved to CSV.")

# Per‑class metrics
per_class_data = []
for i, name in enumerate(class_names):
    mask = y_ext == i
    if mask.sum() > 0:
        acc_cls = accuracy_score(y_ext[mask], y_pred_ext[mask])
        rec = recall_score(y_ext, y_pred_ext, labels=[i], average='macro')
        per_class_data.append([name, acc_cls, rec])
per_class_df = pd.DataFrame(per_class_data, columns=["Class", "Accuracy", "Recall"])
per_class_df.to_csv(tables_dir / "per_class_metrics.csv", index=False)
print("Per‑class metrics saved.")

# Confusion Matrix
cm = confusion_matrix(y_ext, y_pred_ext)
plt.figure(figsize=(10,8))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(ticks=np.arange(len(np.unique(y_ext))), labels=[class_names[i] for i in np.unique(y_ext)], rotation=45)
plt.yticks(ticks=np.arange(len(np.unique(y_ext))), labels=[class_names[i] for i in np.unique(y_ext)])
plt.title('Confusion Matrix (External Set)')
plt.tight_layout()
plt.savefig(plots_dir / "confusion_matrix.png", dpi=150)
plt.show()

# -------------------------------
# 17. Bootstrap Confidence Interval for Accuracy
# -------------------------------
try:
    n_bootstrap = 1000
    boot_accs = []
    for _ in range(n_bootstrap):
        idx = resample(range(len(y_ext)), replace=True, random_state=SEED)
        boot_accs.append(accuracy_score(y_ext[idx], y_pred_ext[idx]))
    ci_low, ci_high = np.percentile(boot_accs, [2.5, 97.5])
    print(f"\nBootstrap 95% CI for Accuracy: [{ci_low:.4f}, {ci_high:.4f}]")
    pd.DataFrame({"bootstrap_accuracy": boot_accs}).to_csv(tables_dir / "bootstrap_accuracies.csv", index=False)
    pd.DataFrame({"ci_lower": [ci_low], "ci_upper": [ci_high]}).to_csv(tables_dir / "accuracy_ci.csv", index=False)
except Exception as e:
    print(f"Bootstrap CI failed: {e}")

# -------------------------------
# 18. Calibration Analysis for Variant C
# -------------------------------
print(f"Expected Calibration Error (ECE): {ece_c:.4f}")
pd.DataFrame({"ECE": [ece_c]}).to_csv(tables_dir / "ece.csv", index=False)

# Calibration curves per class
plt.figure(figsize=(10,8))
for i in np.unique(y_ext):
    class_proba = y_proba_ext[:, i]
    y_binary = (y_ext == i).astype(int)
    fraction_positive, mean_predicted = calibration_curve(y_binary, class_proba, n_bins=10)
    plt.plot(mean_predicted, fraction_positive, marker='s', label=class_names[i])
plt.plot([0,1], [0,1], 'k--', label='Perfect')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.title('Calibration Curves per Class')
plt.tight_layout()
plt.savefig(plots_dir / "calibration_curves.png", dpi=150)
plt.show()

# -------------------------------
# 19. Temperature Scaling Calibration (novelty)
# -------------------------------
try:
    def temperature_scale(logits, T):
        return tf.nn.softmax(logits / T).numpy()

    def nll_loss(T, y_true, logits):
        proba = temperature_scale(logits, T[0])
        return -np.mean(np.log(proba[np.arange(len(y_true)), y_true] + 1e-7))

    logits_val = xgb_model.predict_proba(X_val)
    eps = 1e-7
    logits_val = np.log(np.clip(logits_val, eps, 1-eps))
    result = minimize(nll_loss, x0=[1.0], args=(y_val, logits_val), bounds=[(0.1, 10.0)])
    T_opt = result.x[0]
    print(f"Optimal temperature: {T_opt:.3f}")

    logits_ext = np.log(np.clip(y_proba_ext, eps, 1-eps))
    y_proba_calibrated = temperature_scale(logits_ext, T_opt)
    ece_cal = compute_ece(y_ext, y_proba_calibrated)
    print(f"ECE after temperature scaling: {ece_cal:.4f}")
    pd.DataFrame({"temperature": [T_opt], "ECE_calibrated": [ece_cal]}).to_csv(tables_dir / "temperature_scaling.csv", index=False)
except Exception as e:
    print(f"Temperature scaling failed: {e}")

# -------------------------------
# 20. t‑SNE Visualization of Domain Shift
# -------------------------------
try:
    sample_size = min(1000, len(X_val), len(X_ext))
    X_combined = np.vstack([X_val[:sample_size], X_ext[:sample_size]])
    y_combined = ['lab'] * sample_size + ['field'] * sample_size
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
    embed = tsne.fit_transform(X_combined)
    plt.figure(figsize=(8,6))
    plt.scatter(embed[:sample_size,0], embed[:sample_size,1], c='blue', label='Lab (validation)', alpha=0.6)
    plt.scatter(embed[sample_size:,0], embed[sample_size:,1], c='red', label='Field (external)', alpha=0.6)
    plt.legend()
    plt.title('t‑SNE of Features: Lab vs. Field Domain')
    plt.tight_layout()
    plt.savefig(plots_dir / "tsne_domain_shift.png", dpi=150)
    plt.show()
    np.save(output_dir / "tsne_embeddings.npy", embed)
    pd.DataFrame({"domain": y_combined}).to_csv(tables_dir / "tsne_domain_labels.csv", index=False)
except Exception as e:
    print(f"t‑SNE failed: {e}")

# -------------------------------
# 21. Test-Time Augmentation (TTA) Uncertainty
# -------------------------------
def tta_predict(image, feature_extractor, xgb_model, n_aug=10):
    aug_images = []
    for _ in range(n_aug):
        aug = tf.image.random_flip_left_right(image)
        aug = tf.image.random_brightness(aug, max_delta=0.1)
        aug = tf.image.random_contrast(aug, lower=0.9, upper=1.1)
        aug = tf.image.random_saturation(aug, lower=0.9, upper=1.1)
        aug_images.append(aug)
    aug_images = tf.stack(aug_images)
    features = feature_extractor.predict(aug_images, verbose=0)
    probas = xgb_model.predict_proba(features)
    mean_proba = np.mean(probas, axis=0)
    std_proba = np.std(probas, axis=0)
    return mean_proba, std_proba

try:
    n_samples = 5
    for imgs, lbls in external_ds.take(1):
        external_images = imgs.numpy()
        external_labels = lbls.numpy()
        break

    indices = np.random.choice(len(external_images), min(n_samples, len(external_images)), replace=False)
    tta_results = []
    for idx in indices:
        img = external_images[idx]
        true_label = external_labels[idx]
        img_pp = preprocess_input(img)
        mean_proba, std_proba = tta_predict(img_pp, feature_extractor, xgb_model, n_aug=20)
        pred_class = np.argmax(mean_proba)
        confidence = np.max(mean_proba)
        uncertainty = np.mean(std_proba)
        tta_results.append({
            "sample_idx": idx,
            "true_class": class_names[true_label],
            "pred_class": class_names[pred_class],
            "confidence": confidence,
            "uncertainty": uncertainty
        })
        print(f"Sample {idx}: True={class_names[true_label]}, Pred={class_names[pred_class]}, Confidence={confidence:.3f}, Uncertainty={uncertainty:.3f}")
        print(f"  Mean proba: {dict(zip(class_names, mean_proba.round(3)))}")
        print(f"  Std proba: {std_proba.round(3)}\n")
    pd.DataFrame(tta_results).to_csv(tables_dir / "tta_results.csv", index=False)
except Exception as e:
    print(f"TTA failed: {e}")

# -------------------------------
# 22. Reject Option Based on Uncertainty
# -------------------------------
confidence = np.max(y_proba_ext, axis=1)
uncertainty = 1 - confidence
threshold = 0.3
reject_mask = uncertainty > threshold
accepted_idx = np.where(~reject_mask)[0]
rejected_idx = np.where(reject_mask)[0]

print(f"Rejected {len(rejected_idx)}/{len(y_ext)} samples ({len(rejected_idx)/len(y_ext)*100:.1f}%)")
if len(accepted_idx) > 0:
    acc_accepted = accuracy_score(y_ext[accepted_idx], y_pred_ext[accepted_idx])
    print(f"Accuracy on accepted samples: {acc_accepted:.4f}")
    pd.DataFrame({
        "threshold": [threshold],
        "rejection_rate": [len(rejected_idx)/len(y_ext)],
        "accuracy_accepted": [acc_accepted]
    }).to_csv(tables_dir / "reject_option.csv", index=False)

# -------------------------------
# 23. SHAP Analysis
# -------------------------------
try:
    X_ext_subset = X_ext[:100]
    explainer = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_ext_subset)

    shap.summary_plot(shap_values, X_ext_subset, feature_names=[f'F{i}' for i in range(X_ext_subset.shape[1])], 
                      class_names=class_names, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(plots_dir / "shap_summary.png", dpi=150)
    plt.show()

    shap.summary_plot(shap_values, X_ext_subset, plot_type="bar", class_names=class_names, max_display=20, show=False)
    plt.tight_layout()
    plt.savefig(plots_dir / "shap_bar.png", dpi=150)
    plt.show()
except Exception as e:
    print(f"SHAP analysis failed: {e}")

# -------------------------------
# 24. Efficiency Metrics
# -------------------------------
trainable_params = np.sum([np.prod(v.shape) for v in feature_extractor.trainable_variables])
non_trainable_params = np.sum([np.prod(v.shape) for v in feature_extractor.non_trainable_variables])
print(f"Feature extractor trainable params: {trainable_params:,}")
print(f"Feature extractor non-trainable params: {non_trainable_params:,}")
print(f"Total params: {trainable_params + non_trainable_params:,}")

xgb_size_mb = os.path.getsize(models_dir / "xgb_model.json") / (1024*1024)
print(f"XGBoost model size: {xgb_size_mb:.2f} MB")

# Inference time for a batch of 32 images
batch_images, _ = next(iter(external_ds))
start = time.time()
batch_images_pp = preprocess_input(batch_images)
features = feature_extractor.predict(batch_images_pp, verbose=0)
_ = xgb_model.predict(features)
end = time.time()
inf_time_ms = (end-start)*1000
print(f"Inference time for batch of 32: {inf_time_ms:.2f} ms")
print(f"Per image: {inf_time_ms/32:.2f} ms")
pd.DataFrame({"batch_inference_ms": [inf_time_ms], "per_image_ms": [inf_time_ms/32]}).to_csv(tables_dir / "inference_times.csv", index=False)

print("Approximate FLOPs for MobileNetV2 feature extraction: 300M")

# -------------------------------
# 25. Cross‑Validation on Full Training Features
# -------------------------------
try:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_scores = []
    for train_idx, val_idx in skf.split(X_full, y_full):
        X_tr, X_va = X_full[train_idx], X_full[val_idx]
        y_tr, y_va = y_full[train_idx], y_full[val_idx]
        xgb_cv = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=num_classes,
            random_state=SEED, use_label_encoder=False,
            eval_metric='mlogloss', early_stopping_rounds=10
        )
        xgb_cv.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        pred_va = xgb_cv.predict(X_va)
        cv_scores.append(accuracy_score(y_va, pred_va))
    print(f"5‑fold CV Accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
    pd.DataFrame({"fold": range(1,6), "accuracy": cv_scores}).to_csv(tables_dir / "cross_validation.csv", index=False)
except Exception as e:
    print(f"Cross‑validation failed: {e}")

# -------------------------------
# 26. Results Summary
# -------------------------------
print("\n" + "="*50)
print("FINAL RESULTS")
print("="*50)
print(f"Variant C (Hybrid) External Accuracy: {acc_c:.4f}")
print(f"ECE: {ece_c:.4f}")
print(f"Rejection rate at uncertainty>{threshold}: {len(rejected_idx)/len(y_ext)*100:.1f}%")
if len(accepted_idx) > 0:
    print(f"Accuracy on accepted: {acc_accepted:.4f}")
print(f"Feature extractor params: {trainable_params + non_trainable_params:,}")
print(f"XGBoost model size: {xgb_size_mb:.2f} MB")
print("="*50)

print("\nAblation study results have been saved to outputs/tables/ablation_study.csv")